# 🚗 Comprehensive Car Market Intelligence & Vehicle Pricing Analytics
## A Professional Automotive Business Intelligence Notebook

---

> **Prepared for:** Automotive Manufacturers · Dealership Networks · Market Research Firms · Pricing Analysts · Investors  
> **Methodology:** EDA → Statistical Analysis → Market Intelligence → ML Pricing Prediction → Executive Reporting  
> **Environment:** Kaggle Notebook | Python 3.10+

---

## 📌 Project Overview

This notebook delivers a **full-stack automotive market intelligence workflow** — from raw dataset ingestion through executive-level business recommendations. It mirrors the analytical depth of a professional consulting engagement, covering:

| Analysis Layer | Coverage |
|---|---|
| 🔍 Data Intelligence | Dynamic inspection, quality assessment, cleaning |
| 🏭 Manufacturer Intelligence | Rankings, portfolio, market share, pricing |
| 🚘 Model Intelligence | Distribution, pricing tiers, segment analysis |
| 🔧 Trim Intelligence | Trim depth, pricing spread, premium identification |
| 💰 Pricing Intelligence | MSRP vs Invoice, margins, segmentation |
| 🌍 Market Intelligence | Competitive landscape, leaders, concentration |
| 📈 Statistical Analysis | Correlations, distributions, hypothesis testing |
| 🤖 ML Prediction | MSRP prediction with feature importance |
| 🧠 Insights Engine | Automated narrative generation |
| 📋 Executive Report | Findings, recommendations, opportunities |

## 🎯 Business Objectives

1. Identify **market leaders** and competitive positioning across manufacturers
2. Quantify **pricing gaps** between MSRP and Invoice for negotiation intelligence
3. Segment the market into **Budget / Mid-Range / Premium / Luxury** tiers
4. Build a **predictive model** to estimate vehicle MSRP from available features
5. Generate **actionable recommendations** for dealerships, manufacturers, and investors


## 2. 🛠️ Environment Setup & Library Imports

In [ ]:
# ── Core Libraries ────────────────────────────────────────────────────────────
import os, glob, re, warnings, itertools
import numpy  as np
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot    as plt
import matplotlib.gridspec  as gridspec
import matplotlib.ticker    as mticker
import matplotlib.patches   as mpatches
from   matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import plotly.express        as px
import plotly.graph_objects  as go
from   plotly.subplots import make_subplots

# ── Statistics ────────────────────────────────────────────────────────────────
from scipy import stats
from scipy.stats import pearsonr, spearmanr, shapiro, normaltest

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.ensemble         import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model     import LinearRegression, Ridge
from sklearn.preprocessing    import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.model_selection  import train_test_split, cross_val_score, KFold
from sklearn.metrics          import r2_score, mean_absolute_error, mean_squared_error
from sklearn.impute           import SimpleImputer
from sklearn.pipeline         import Pipeline
from sklearn.inspection       import permutation_importance

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.width', 200)

# ── Global Aesthetic Theme ────────────────────────────────────────────────────
BG_DARK    = '#0D1117'
BG_PANEL   = '#161B22'
BG_BORDER  = '#30363D'
TEXT_LIGHT = '#E6EDF3'
TEXT_MUTED = '#8B949E'

CAR_RED    = '#E63946'
CAR_GOLD   = '#FFB703'
CAR_BLUE   = '#023E8A'
CAR_TEAL   = '#06D6A0'
CAR_ORANGE = '#FB8500'
CAR_PURPLE = '#7209B7'
CAR_SILVER = '#ADB5BD'

PALETTE_MAIN = [CAR_RED, CAR_GOLD, CAR_BLUE, CAR_TEAL, CAR_ORANGE,
                CAR_PURPLE, CAR_SILVER, '#4CC9F0', '#F72585', '#3A0CA3',
                '#4361EE', '#560BAD', '#B5179E', '#7FBA00', '#00B4D8']

plt.rcParams.update({
    'figure.facecolor' : BG_DARK,
    'axes.facecolor'   : BG_PANEL,
    'axes.edgecolor'   : BG_BORDER,
    'axes.labelcolor'  : TEXT_LIGHT,
    'axes.titlecolor'  : TEXT_LIGHT,
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
    'axes.titleweight' : 'bold',
    'axes.grid'        : True,
    'grid.color'       : BG_BORDER,
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.5,
    'xtick.color'      : TEXT_MUTED,
    'ytick.color'      : TEXT_MUTED,
    'legend.facecolor' : BG_PANEL,
    'legend.edgecolor' : BG_BORDER,
    'text.color'       : TEXT_LIGHT,
    'font.family'      : 'DejaVu Sans',
    'figure.titlesize' : 16,
    'figure.titleweight': 'bold',
})

print("✅ Environment configured.")
print(f"   pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}")


## 3. 🔍 Data Discovery & Loading

We dynamically scan the Kaggle input directory and load all available CSV files. No column names are hardcoded.


In [ ]:
# ── Step 1: Discover all files ────────────────────────────────────────────────
print("=" * 65)
print("  📁  KAGGLE INPUT DIRECTORY — FILE DISCOVERY")
print("=" * 65)

all_files = []
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        full_path = os.path.join(dirname, filename)
        size_kb   = os.path.getsize(full_path) / 1024
        all_files.append((full_path, filename, f"{size_kb:,.1f} KB"))
        print(f"  📄  {full_path:<70}  {size_kb:>10,.1f} KB")

csv_files = [f for f, _, __ in all_files if f.endswith('.csv')]
print(f"\n  Total files   : {len(all_files)}")
print(f"  CSV files     : {len(csv_files)}")


In [ ]:
# ── Step 2: Load dataset ─────────────────────────────────────────────────────
def load_best_csv(csv_list):
    """Load the largest CSV file (most likely the main dataset)."""
    if not csv_list:
        raise FileNotFoundError("No CSV files found in /kaggle/input")
    # Sort by file size descending — take the primary dataset
    sizes = [(f, os.path.getsize(f)) for f in csv_list]
    sizes.sort(key=lambda x: x[1], reverse=True)
    frames = {}
    for path, sz in sizes:
        name = os.path.splitext(os.path.basename(path))[0]
        try:
            df = pd.read_csv(path, low_memory=False)
            print(f"  ✅  {name}: {df.shape[0]:,} rows × {df.shape[1]} cols  ({sz/1024:,.1f} KB)")
            frames[name] = df
        except Exception as e:
            print(f"  ❌  {path}: {e}")
    return frames

print("\n" + "=" * 65)
print("  📊  LOADING DATASETS")
print("=" * 65 + "\n")

datasets = load_best_csv(csv_files)

# Use the largest dataset as primary
if datasets:
    primary_name = list(datasets.keys())[0]
    df_raw = datasets[primary_name].copy()
    print(f"\n  Primary dataset  : '{primary_name}'")
    print(f"  Shape            : {df_raw.shape}")
else:
    raise RuntimeError("No datasets could be loaded.")


In [ ]:
# ── Step 3: Inspect columns & dtypes ─────────────────────────────────────────
print("=" * 65)
print("  🔎  COLUMN INSPECTION")
print("=" * 65)
print(f"  {'#':<4} {'Column Name':<40} {'Dtype':<15} {'Non-Null':<12} {'Unique'}")
print("  " + "-" * 80)
for i, col in enumerate(df_raw.columns, 1):
    dtype     = str(df_raw[col].dtype)
    non_null  = df_raw[col].notna().sum()
    n_unique  = df_raw[col].nunique()
    print(f"  {i:<4} {col:<40} {dtype:<15} {non_null:<12,} {n_unique:,}")

# Auto-detect numeric and categorical columns
NUM_COLS  = df_raw.select_dtypes(include=np.number).columns.tolist()
CAT_COLS  = df_raw.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"\n  Numeric columns     : {len(NUM_COLS)}")
print(f"  Categorical columns : {len(CAT_COLS)}")


In [ ]:
# ── Step 4: Dataset preview ───────────────────────────────────────────────────
print("\n── First 5 rows ──")
display(df_raw.head())
print("\n── Random sample (5 rows) ──")
display(df_raw.sample(min(5, len(df_raw)), random_state=42))
print("\n── Descriptive statistics ──")
display(df_raw.describe(include='all').T)


In [ ]:
# ── Step 5: Missing value map ─────────────────────────────────────────────────
missing = df_raw.isna().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count' : missing,
    'Missing %'     : missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print("\n  📋  MISSING VALUE REPORT")
if missing_df.empty:
    print("  ✅  No missing values detected!")
else:
    display(missing_df)

dupes = df_raw.duplicated().sum()
print(f"\n  Duplicate rows : {dupes:,}")

# Visualise missing values
if not missing_df.empty:
    fig, ax = plt.subplots(figsize=(12, max(4, len(missing_df)*0.4 + 1)))
    fig.suptitle("Missing Value Analysis", fontsize=14, color=CAR_GOLD)
    bars = ax.barh(missing_df.index, missing_df['Missing %'],
                   color=CAR_RED, edgecolor=BG_DARK, linewidth=0.5, alpha=0.85)
    ax.set_xlabel("Missing %")
    ax.set_title("Columns with Missing Data")
    for bar in bars:
        w = bar.get_width()
        ax.text(w + 0.3, bar.get_y() + bar.get_height()/2, f'{w:.1f}%',
                va='center', fontsize=9, color=TEXT_LIGHT)
    plt.tight_layout()
    plt.savefig('/kaggle/working/missing_values.png', dpi=140, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()


## 4. 🧹 Data Cleaning & Preprocessing

**Strategy:**  
1. Standardise column names (snake_case)  
2. Remove full duplicates  
3. Convert numeric columns stored as strings  
4. Handle missing values (median for numeric, mode/constant for categorical)  
5. Detect and flag price outliers  
6. Generate a Data Quality Score


In [ ]:
# ── Clean column names ────────────────────────────────────────────────────────
def clean_cols(df):
    df.columns = (df.columns
                    .str.strip()
                    .str.lower()
                    .str.replace(r'[\s]+', '_', regex=True)
                    .str.replace(r'[^a-z0-9_]', '', regex=True))
    return df

df = clean_cols(df_raw.copy())
print("  ✅  Column names standardised.")
print("  New columns:", list(df.columns))


In [ ]:
# ── Remove duplicates ─────────────────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates()
after  = len(df)
print(f"  Duplicates removed : {before - after:,}  (Remaining rows: {after:,})")

# ── Convert object → numeric where possible ───────────────────────────────────
def coerce_numeric(df, exclude_hint=None):
    exclude_hint = exclude_hint or []
    converted = []
    for c in df.select_dtypes(include='object').columns:
        if any(h in c for h in exclude_hint):
            continue
        # Strip common currency / comma characters
        cleaned = df[c].astype(str).str.replace(r'[\$,\s]', '', regex=True)
        numeric = pd.to_numeric(cleaned, errors='coerce')
        orig_nn = df[c].notna().sum()
        conv_nn = numeric.notna().sum()
        if conv_nn >= 0.5 * orig_nn and conv_nn > 0:
            df[c] = numeric
            converted.append(c)
    return df, converted

TEXT_HINTS = ['name', 'make', 'model', 'trim', 'desc', 'type', 'fuel', 'drive',
              'transmission', 'body', 'color', 'style', 'cat', 'class', 'origin']
df, converted = coerce_numeric(df, exclude_hint=TEXT_HINTS)
print(f"  Columns coerced to numeric : {converted}")

# ── Re-detect column types ────────────────────────────────────────────────────
NUM_COLS = df.select_dtypes(include=np.number).columns.tolist()
CAT_COLS = df.select_dtypes(include=['object','category']).columns.tolist()
print(f"  Numeric columns     : {NUM_COLS}")
print(f"  Categorical columns : {CAT_COLS}")


In [ ]:
# ── Missing value imputation ──────────────────────────────────────────────────
for c in NUM_COLS:
    if df[c].isna().any():
        median_val = df[c].median()
        df[c].fillna(median_val, inplace=True)
        print(f"  Filled numeric  '{c}' with median={median_val:,.2f}")

for c in CAT_COLS:
    if df[c].isna().any():
        mode_val = df[c].mode()[0] if not df[c].mode().empty else 'Unknown'
        df[c].fillna(mode_val, inplace=True)
        print(f"  Filled categoric'{c}' with mode='{mode_val}'")

print("\n  ✅  Missing value imputation complete.")


In [ ]:
# ── Identify pricing columns dynamically ─────────────────────────────────────
def find_col(df, *candidates):
    """Return first matching column or None."""
    for c in candidates:
        if c in df.columns:
            return c
        # fuzzy match
        matches = [col for col in df.columns if c in col]
        if matches:
            return matches[0]
    return None

MSRP_COL    = find_col(df, 'msrp', 'trim_msrp', 'price', 'retail_price', 'list_price')
INVOICE_COL = find_col(df, 'invoice', 'trim_invoice', 'invoice_price', 'dealer_price')
MAKE_COL    = find_col(df, 'make_name', 'make', 'manufacturer', 'brand')
MODEL_COL   = find_col(df, 'model_name', 'model', 'vehicle_model')
TRIM_COL    = find_col(df, 'trim_name', 'trim', 'trim_level', 'variant')
YEAR_COL    = find_col(df, 'trim_year', 'year', 'model_year', 'vehicle_year')
DESC_COL    = find_col(df, 'trim_description', 'description', 'desc')
MAKE_ID     = find_col(df, 'make_id')
MODEL_ID    = find_col(df, 'model_id')
TRIM_ID     = find_col(df, 'trim_id')

print("  📌  Identified Key Columns:")
for label, c in [('MSRP', MSRP_COL), ('Invoice', INVOICE_COL), ('Make', MAKE_COL),
                  ('Model', MODEL_COL), ('Trim', TRIM_COL), ('Year', YEAR_COL),
                  ('Description', DESC_COL)]:
    status = '✅' if c else '⚠️ NOT FOUND'
    print(f"  {status}  {label:<15}: {c}")


In [ ]:
# ── Outlier detection (IQR method) for pricing columns ───────────────────────
def detect_outliers_iqr(series, label):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR    = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_outliers = ((series < lower) | (series > upper)).sum()
    print(f"  {label:<20}: Q1={Q1:>12,.0f}  Q3={Q3:>12,.0f}  IQR={IQR:>10,.0f}  Outliers={n_outliers:,}")
    return lower, upper

print("  📊  OUTLIER REPORT (IQR Method)")
pricing_cols = [c for c in [MSRP_COL, INVOICE_COL] if c]
for c in pricing_cols:
    detect_outliers_iqr(df[c].dropna(), c)

# ── Data Quality Score ────────────────────────────────────────────────────────
total_cells   = df.shape[0] * df.shape[1]
missing_after = df.isna().sum().sum()
dupe_after    = df.duplicated().sum()
dq_score      = max(0, 100 - (missing_after / total_cells * 100) - (dupe_after / len(df) * 10))

print(f"\n  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  📋  DATA QUALITY SCORE : {dq_score:.1f} / 100")
print(f"  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  Final dataset shape   : {df.shape}")
print(f"  Missing cells (after) : {missing_after:,}")
print(f"  Duplicate rows (after): {dupe_after:,}")


## 5. 📊 Executive Summary Dashboard

High-level KPIs giving an instant snapshot of the automotive market dataset.


In [ ]:
# ── KPI Computation ───────────────────────────────────────────────────────────
kpis = {}

kpis['Total Records']        = f"{len(df):,}"
kpis['Unique Manufacturers'] = f"{df[MAKE_COL].nunique():,}"   if MAKE_COL  else "N/A"
kpis['Unique Models']        = f"{df[MODEL_COL].nunique():,}"  if MODEL_COL else "N/A"
kpis['Unique Trims']         = f"{df[TRIM_COL].nunique():,}"   if TRIM_COL  else "N/A"

if MSRP_COL:
    msrp_clean = df[MSRP_COL].dropna()
    kpis['Avg MSRP']         = f"${msrp_clean.mean():>12,.0f}"
    kpis['Median MSRP']      = f"${msrp_clean.median():>12,.0f}"
    kpis['Min MSRP']         = f"${msrp_clean.min():>12,.0f}"
    kpis['Max MSRP']         = f"${msrp_clean.max():>12,.0f}"

if INVOICE_COL:
    inv_clean = df[INVOICE_COL].dropna()
    kpis['Avg Invoice']      = f"${inv_clean.mean():>12,.0f}"
    kpis['Median Invoice']   = f"${inv_clean.median():>12,.0f}"

if MSRP_COL and INVOICE_COL:
    gap = df[MSRP_COL] - df[INVOICE_COL]
    kpis['Avg Price Gap']    = f"${gap.mean():>12,.0f}"
    kpis['Max Price Gap']    = f"${gap.max():>12,.0f}"

if YEAR_COL:
    kpis['Year Range']       = f"{int(df[YEAR_COL].min())} – {int(df[YEAR_COL].max())}"

print("\n  ═══════════════════════════════════════════════════")
print("    🏎️   AUTOMOTIVE MARKET — EXECUTIVE KPI DASHBOARD")
print("  ═══════════════════════════════════════════════════")
for k, v in kpis.items():
    print(f"  {'•'} {k:<30} {v}")
print("  ═══════════════════════════════════════════════════")


In [ ]:
# ── Executive KPI Visual Dashboard ───────────────────────────────────────────
fig = plt.figure(figsize=(20, 6))
fig.patch.set_facecolor(BG_DARK)
fig.suptitle('🚗  Automotive Market — Executive KPI Dashboard', 
             color=CAR_GOLD, fontsize=16, fontweight='bold', y=1.02)

kpi_items = [(k, v) for k, v in kpis.items()]
n_kpis    = len(kpi_items)
cols      = min(n_kpis, 5)
rows      = (n_kpis + cols - 1) // cols

for idx, (key, val) in enumerate(kpi_items):
    ax = fig.add_subplot(rows, cols, idx + 1)
    ax.set_facecolor('#1C2128')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis('off')
    colour = PALETTE_MAIN[idx % len(PALETTE_MAIN)]
    # Card border
    rect = mpatches.FancyBboxPatch((0.03, 0.05), 0.94, 0.9,
                                    boxstyle="round,pad=0.03",
                                    linewidth=2, edgecolor=colour, facecolor='#1C2128')
    ax.add_patch(rect)
    ax.text(0.5, 0.68, str(val), ha='center', va='center',
            fontsize=15, fontweight='bold', color=colour, transform=ax.transAxes)
    ax.text(0.5, 0.25, key, ha='center', va='center',
            fontsize=8.5, color=TEXT_MUTED, transform=ax.transAxes, wrap=True)

plt.tight_layout(pad=1.5)
plt.savefig('/kaggle/working/executive_kpis.png', dpi=150, bbox_inches='tight', facecolor=BG_DARK)
plt.show()
print("✅ Executive KPI dashboard rendered.")


## 6. 🏭 Manufacturer Intelligence & Analytics

Deep-dive into manufacturer-level market representation, pricing positioning, and portfolio breadth.


In [ ]:
if MAKE_COL:
    # ── Manufacturer ranking ──────────────────────────────────────────────────
    mfr_counts = df[MAKE_COL].value_counts().reset_index()
    mfr_counts.columns = ['Manufacturer', 'Vehicle Count']
    mfr_counts['Market Share %'] = (mfr_counts['Vehicle Count'] / mfr_counts['Vehicle Count'].sum() * 100).round(2)
    mfr_counts['Cumulative %']   = mfr_counts['Market Share %'].cumsum().round(2)
    mfr_counts.index = range(1, len(mfr_counts) + 1)

    print("  🏆  TOP 20 MANUFACTURERS BY VEHICLE COUNT")
    print("  " + "─" * 65)
    display(mfr_counts.head(20))
else:
    print("  ⚠️  Make column not found — skipping manufacturer analysis.")


In [ ]:
if MAKE_COL:
    fig, axes = plt.subplots(1, 2, figsize=(22, 8))
    fig.suptitle('🏭  Manufacturer Market Intelligence', color=CAR_GOLD, fontsize=15)

    # Top 20 manufacturers — horizontal bar
    ax = axes[0]
    top20 = mfr_counts.head(20)
    colours = [PALETTE_MAIN[i % len(PALETTE_MAIN)] for i in range(len(top20))]
    bars = ax.barh(range(len(top20))[::-1], top20['Vehicle Count'],
                   color=colours, edgecolor=BG_DARK, linewidth=0.4)
    ax.set_yticks(range(len(top20))[::-1])
    ax.set_yticklabels(top20['Manufacturer'], fontsize=9)
    ax.set_title('Top 20 Manufacturers by Vehicle Count')
    ax.set_xlabel('Number of Vehicles')
    for bar in bars:
        w = bar.get_width()
        ax.text(w + top20['Vehicle Count'].max()*0.01, bar.get_y() + bar.get_height()/2,
                f'{w:,.0f}', va='center', fontsize=8, color=TEXT_LIGHT)

    # Market share — Pareto chart
    ax2 = axes[1]
    top15 = mfr_counts.head(15)
    ax2.bar(range(len(top15)), top15['Market Share %'],
            color=PALETTE_MAIN[:len(top15)], edgecolor=BG_DARK, linewidth=0.4)
    ax2.set_xticks(range(len(top15)))
    ax2.set_xticklabels(top15['Manufacturer'], rotation=45, ha='right', fontsize=8)
    ax2.set_ylabel('Market Share %', color=TEXT_LIGHT)
    ax2.set_title('Manufacturer Market Share & Cumulative Coverage')
    ax_r = ax2.twinx()
    ax_r.plot(range(len(top15)), top15['Cumulative %'], color=CAR_GOLD,
              marker='o', markersize=4, linewidth=2, label='Cumulative %')
    ax_r.set_ylabel('Cumulative %', color=CAR_GOLD)
    ax_r.tick_params(axis='y', colors=CAR_GOLD)
    ax_r.axhline(80, color=CAR_RED, linestyle='--', linewidth=1, alpha=0.7, label='80% line')
    ax_r.legend(loc='upper left', fontsize=8)

    plt.tight_layout()
    plt.savefig('/kaggle/working/manufacturer_market_share.png', dpi=140, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()


In [ ]:
# ── Plotly Treemap — Manufacturer Portfolio ───────────────────────────────────
if MAKE_COL:
    fig_tree = px.treemap(
        mfr_counts.head(30),
        path=['Manufacturer'],
        values='Vehicle Count',
        color='Market Share %',
        color_continuous_scale='RdYlGn',
        title='🏭  Manufacturer Portfolio Treemap — Vehicle Count by Brand',
        template='plotly_dark'
    )
    fig_tree.update_layout(
        font=dict(family='Arial', size=13),
        title_font=dict(size=16, color=CAR_GOLD),
        margin=dict(t=60, l=10, r=10, b=10),
        height=520
    )
    fig_tree.show()


In [ ]:
# ── Manufacturer Pricing Analysis ─────────────────────────────────────────────
if MAKE_COL and MSRP_COL:
    mfr_price = df.groupby(MAKE_COL)[MSRP_COL].agg(
        Avg_MSRP='mean', Median_MSRP='median',
        Min_MSRP='min',  Max_MSRP='max', Count='count'
    ).round(0).sort_values('Avg_MSRP', ascending=False)
    mfr_price = mfr_price[mfr_price['Count'] >= 3]   # at least 3 models
    mfr_price.index.name = 'Manufacturer'
    mfr_price = mfr_price.reset_index()

    print("  💰  MANUFACTURER PRICING INTELLIGENCE (Top 25 by Avg MSRP)")
    display(mfr_price.head(25))

    # ── Premium vs Budget segmentation ────────────────────────────────────────
    overall_median = df[MSRP_COL].median()
    mfr_price['Segment'] = mfr_price['Avg_MSRP'].apply(
        lambda x: 'Luxury' if x > overall_median * 2 else
                  'Premium' if x > overall_median * 1.3 else
                  'Mid-Range' if x >= overall_median * 0.7 else 'Budget'
    )
    seg_summary = mfr_price['Segment'].value_counts()
    print("\n  📊  MARKET SEGMENT DISTRIBUTION (by Manufacturer Avg MSRP):")
    for seg, cnt in seg_summary.items():
        print(f"  {'▶'} {seg:<12}: {cnt:>4} manufacturers")


In [ ]:
if MAKE_COL and MSRP_COL:
    fig, axes = plt.subplots(1, 2, figsize=(22, 8))
    fig.suptitle('💰  Manufacturer Pricing Intelligence', color=CAR_GOLD, fontsize=15)

    # Average MSRP by manufacturer (top 20)
    ax = axes[0]
    top20_price = mfr_price.head(20)
    seg_colour_map = {'Luxury': CAR_GOLD, 'Premium': CAR_ORANGE, 'Mid-Range': CAR_TEAL, 'Budget': CAR_SILVER}
    bar_colours = [seg_colour_map.get(s, CAR_BLUE) for s in top20_price['Segment']]
    bars = ax.barh(range(len(top20_price))[::-1], top20_price['Avg_MSRP'],
                   color=bar_colours, edgecolor=BG_DARK, linewidth=0.4)
    ax.set_yticks(range(len(top20_price))[::-1])
    ax.set_yticklabels(top20_price['Manufacturer'], fontsize=9)
    ax.set_title('Average MSRP by Manufacturer (Top 20)')
    ax.set_xlabel('Average MSRP ($)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    legend_patches = [mpatches.Patch(color=v, label=k) for k, v in seg_colour_map.items()]
    ax.legend(handles=legend_patches, loc='lower right', fontsize=8)

    # Segment pie chart
    ax = axes[1]
    seg_c = [seg_colour_map.get(s, CAR_BLUE) for s in seg_summary.index]
    wedge_props = dict(edgecolor=BG_DARK, linewidth=2)
    ax.pie(seg_summary.values, labels=seg_summary.index, autopct='%1.1f%%',
           colors=seg_c, wedgeprops=wedge_props,
           textprops={'fontsize': 11, 'color': TEXT_LIGHT})
    ax.set_title('Market Segmentation by Manufacturer Pricing')

    plt.tight_layout()
    plt.savefig('/kaggle/working/manufacturer_pricing.png', dpi=140, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()


In [ ]:
# ── Plotly Sunburst — Manufacturer × Segment ──────────────────────────────────
if MAKE_COL and MSRP_COL:
    fig_sun = px.sunburst(
        mfr_price.head(40),
        path=['Segment', 'Manufacturer'],
        values='Count',
        color='Avg_MSRP',
        color_continuous_scale='RdYlGn',
        title='🌐  Manufacturer Market Map — Segment × Brand',
        template='plotly_dark'
    )
    fig_sun.update_layout(height=550, title_font=dict(size=15, color=CAR_GOLD),
                          margin=dict(t=60, l=10, r=10, b=10))
    fig_sun.show()


## 7. 🚘 Vehicle Model Intelligence & Analytics

In [ ]:
if MODEL_COL:
    model_counts = df[MODEL_COL].value_counts().reset_index()
    model_counts.columns = ['Model', 'Count']
    print("  🚘  TOP 30 VEHICLE MODELS BY FREQUENCY")
    display(model_counts.head(30))

    # Models per manufacturer
    if MAKE_COL:
        models_per_make = df.groupby(MAKE_COL)[MODEL_COL].nunique().sort_values(ascending=False)
        print("\n  🏭  MODELS PER MANUFACTURER (Most Diversified)")
        display(models_per_make.head(20).reset_index().rename(
            columns={MAKE_COL: 'Manufacturer', MODEL_COL: 'Unique Models'}))
else:
    print("  ⚠️  Model column not found — skipping model analysis.")


In [ ]:
if MODEL_COL:
    fig, axes = plt.subplots(1, 2, figsize=(22, 8))
    fig.suptitle('🚘  Vehicle Model Analytics', color=CAR_GOLD, fontsize=15)

    # Top 20 models by count
    ax = axes[0]
    top20_m = model_counts.head(20)
    bars = ax.bar(range(len(top20_m)), top20_m['Count'],
                  color=PALETTE_MAIN[:len(top20_m)], edgecolor=BG_DARK, linewidth=0.4)
    ax.set_xticks(range(len(top20_m)))
    ax.set_xticklabels(top20_m['Model'], rotation=45, ha='right', fontsize=8)
    ax.set_title('Top 20 Models by Trim Count')
    ax.set_ylabel('Number of Trims')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, f'{h:.0f}',
                ha='center', va='bottom', fontsize=7, color=TEXT_LIGHT)

    # Models per make — top 15
    ax = axes[1]
    if MAKE_COL:
        top15_div = models_per_make.head(15)
        ax.barh(range(len(top15_div))[::-1], top15_div.values,
                color=CAR_TEAL, edgecolor=BG_DARK, linewidth=0.4)
        ax.set_yticks(range(len(top15_div))[::-1])
        ax.set_yticklabels(top15_div.index, fontsize=9)
        ax.set_title('Most Diversified Manufacturers (Unique Models)')
        ax.set_xlabel('Number of Unique Models')
    else:
        ax.text(0.5, 0.5, 'Make data unavailable', ha='center', va='center',
                transform=ax.transAxes, color=TEXT_MUTED)

    plt.tight_layout()
    plt.savefig('/kaggle/working/model_analytics.png', dpi=140, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()


In [ ]:
# ── Model Pricing Analysis ────────────────────────────────────────────────────
if MODEL_COL and MSRP_COL and MAKE_COL:
    model_price = df.groupby([MAKE_COL, MODEL_COL])[MSRP_COL].agg(
        Avg_MSRP='mean', Min_MSRP='min', Max_MSRP='max', Count='count'
    ).round(0).reset_index()
    model_price.columns = ['Make', 'Model', 'Avg_MSRP', 'Min_MSRP', 'Max_MSRP', 'Count']
    model_price['Price_Range'] = model_price['Max_MSRP'] - model_price['Min_MSRP']

    print("  💰  HIGHEST PRICED MODELS (Avg MSRP)")
    display(model_price.nlargest(15, 'Avg_MSRP'))
    print("\n  💰  LOWEST PRICED MODELS (Avg MSRP)")
    display(model_price.nsmallest(15, 'Avg_MSRP'))
    print("\n  📊  WIDEST TRIM PRICE RANGE (Max - Min MSRP)")
    display(model_price.nlargest(15, 'Price_Range'))


In [ ]:
if MODEL_COL and MSRP_COL and MAKE_COL:
    fig, axes = plt.subplots(1, 2, figsize=(22, 8))
    fig.suptitle('🚘  Model Pricing Intelligence', color=CAR_GOLD, fontsize=15)

    # Highest avg MSRP models
    ax = axes[0]
    top15_hi = model_price.nlargest(15, 'Avg_MSRP')
    labels   = top15_hi.apply(lambda r: f"{r['Make']} {r['Model']}"[:25], axis=1)
    bars = ax.barh(range(len(top15_hi))[::-1], top15_hi['Avg_MSRP'],
                   color=CAR_GOLD, edgecolor=BG_DARK, linewidth=0.4)
    ax.set_yticks(range(len(top15_hi))[::-1])
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_title('Top 15 Highest Avg MSRP Models')
    ax.set_xlabel('Average MSRP ($)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    # Widest price range
    ax = axes[1]
    top15_range = model_price.nlargest(15, 'Price_Range')
    labels_r    = top15_range.apply(lambda r: f"{r['Make']} {r['Model']}"[:25], axis=1)
    ax.barh(range(len(top15_range))[::-1], top15_range['Price_Range'],
            color=CAR_ORANGE, edgecolor=BG_DARK, linewidth=0.4)
    ax.set_yticks(range(len(top15_range))[::-1])
    ax.set_yticklabels(labels_r, fontsize=8)
    ax.set_title('Widest Trim Price Range Models')
    ax.set_xlabel('Price Range ($)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    plt.tight_layout()
    plt.savefig('/kaggle/working/model_pricing.png', dpi=140, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()


## 8. 🔧 Trim-Level Intelligence & Analytics

In [ ]:
if TRIM_COL:
    trim_counts = df[TRIM_COL].value_counts().reset_index()
    trim_counts.columns = ['Trim', 'Count']
    print("  🔧  TOP 30 MOST COMMON TRIMS")
    display(trim_counts.head(30))

    # Trim pricing
    if MSRP_COL:
        trim_price = df.groupby(TRIM_COL)[MSRP_COL].agg(
            Avg_MSRP='mean', Count='count').sort_values('Avg_MSRP', ascending=False)
        trim_price = trim_price[trim_price['Count'] >= 3].reset_index()
        print("\n  💰  HIGHEST PRICED TRIMS (Avg MSRP, min 3 records)")
        display(trim_price.head(20))
else:
    print("  ⚠️  Trim column not found — skipping trim analysis.")


In [ ]:
if TRIM_COL and MSRP_COL:
    fig, axes = plt.subplots(1, 2, figsize=(22, 8))
    fig.suptitle('🔧  Trim Analytics', color=CAR_GOLD, fontsize=15)

    # Trim frequency
    ax = axes[0]
    top25_trim = trim_counts.head(25)
    ax.bar(range(len(top25_trim)), top25_trim['Count'],
           color=PALETTE_MAIN[:len(top25_trim)], edgecolor=BG_DARK, linewidth=0.3)
    ax.set_xticks(range(len(top25_trim)))
    ax.set_xticklabels(top25_trim['Trim'], rotation=60, ha='right', fontsize=7)
    ax.set_title('Top 25 Most Common Trims')
    ax.set_ylabel('Frequency')

    # Highest priced trims
    ax = axes[1]
    top15_tp = trim_price.head(15)
    ax.barh(range(len(top15_tp))[::-1], top15_tp['Avg_MSRP'],
            color=CAR_PURPLE, edgecolor=BG_DARK, linewidth=0.4)
    ax.set_yticks(range(len(top15_tp))[::-1])
    ax.set_yticklabels(top15_tp[TRIM_COL], fontsize=9)
    ax.set_title('Highest Avg MSRP Trims (min 3 records)')
    ax.set_xlabel('Average MSRP ($)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    plt.tight_layout()
    plt.savefig('/kaggle/working/trim_analytics.png', dpi=140, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()


In [ ]:
# ── Trim Year Analysis ────────────────────────────────────────────────────────
if YEAR_COL:
    year_dist = df[YEAR_COL].dropna().value_counts().sort_index()
    print(f"  📅  TRIM YEAR RANGE: {int(year_dist.index.min())} – {int(year_dist.index.max())}")
    display(year_dist.reset_index().rename(columns={YEAR_COL: 'Year', 'count': 'Count'}).head(30))

    fig, axes = plt.subplots(1, 2, figsize=(20, 6))
    fig.suptitle('📅  Trim Year Trends', color=CAR_GOLD, fontsize=14)

    ax = axes[0]
    ax.bar(year_dist.index, year_dist.values, color=CAR_BLUE, edgecolor=BG_DARK, linewidth=0.3, alpha=0.85)
    ax.set_title('Vehicle Count by Model Year')
    ax.set_xlabel('Year'); ax.set_ylabel('Count')

    ax = axes[1]
    if MSRP_COL:
        year_msrp = df.groupby(YEAR_COL)[MSRP_COL].mean().dropna()
        ax.plot(year_msrp.index, year_msrp.values, color=CAR_GOLD, linewidth=2, marker='o', markersize=4)
        ax.fill_between(year_msrp.index, year_msrp.values, alpha=0.2, color=CAR_GOLD)
        ax.set_title('Average MSRP Trend by Year')
        ax.set_xlabel('Year'); ax.set_ylabel('Average MSRP ($)')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    else:
        ax.text(0.5, 0.5, 'MSRP not available', ha='center', va='center',
                transform=ax.transAxes, color=TEXT_MUTED)

    plt.tight_layout()
    plt.savefig('/kaggle/working/year_trends.png', dpi=140, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()
else:
    print("  ⚠️  Year column not found — skipping year analysis.")


## 9. 💰 Pricing Intelligence & Market Segmentation

Advanced pricing analytics including MSRP vs Invoice analysis, price gap intelligence, and four-tier market segmentation.


In [ ]:
if MSRP_COL:
    msrp = df[MSRP_COL].dropna()

    # Distribution statistics
    stats_dict = {
        'Mean'     : msrp.mean(),
        'Median'   : msrp.median(),
        'Std Dev'  : msrp.std(),
        'Skewness' : msrp.skew(),
        'Kurtosis' : msrp.kurtosis(),
        'Min'      : msrp.min(),
        'P10'      : msrp.quantile(0.10),
        'P25'      : msrp.quantile(0.25),
        'P75'      : msrp.quantile(0.75),
        'P90'      : msrp.quantile(0.90),
        'Max'      : msrp.max(),
    }
    print("  📊  MSRP DISTRIBUTION STATISTICS")
    for k, v in stats_dict.items():
        print(f"  {'•'} {k:<12}: ${v:>14,.0f}" if k not in ('Skewness','Kurtosis')
              else f"  {'•'} {k:<12}: {v:>14.4f}")

    # ── Price tier segmentation ───────────────────────────────────────────────
    def price_tier(price):
        if price < 20_000:             return '💚 Budget (<$20K)'
        elif price < 40_000:           return '🔵 Mid-Range ($20K-$40K)'
        elif price < 70_000:           return '🟠 Premium ($40K-$70K)'
        elif price < 100_000:          return '🟡 Luxury ($70K-$100K)'
        else:                          return '💎 Ultra-Luxury (>$100K)'

    df['price_tier'] = df[MSRP_COL].apply(price_tier)
    tier_counts = df['price_tier'].value_counts()
    tier_pct    = (tier_counts / tier_counts.sum() * 100).round(1)

    print("\n  📊  MARKET PRICE TIER SEGMENTATION")
    for tier, cnt in tier_counts.items():
        pct = tier_pct[tier]
        bar_len = int(pct / 2)
        print(f"  {tier:<35} {cnt:>7,} vehicles  ({pct:>5.1f}%)  {'█'*bar_len}")
else:
    print("  ⚠️  MSRP column not found — skipping pricing analysis.")


In [ ]:
if MSRP_COL:
    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    fig.suptitle('💰  Pricing Intelligence Dashboard', color=CAR_GOLD, fontsize=16)
    axes = axes.flatten()

    # MSRP histogram with KDE
    ax = axes[0]
    msrp_plot = df[MSRP_COL].dropna()
    msrp_plot_clip = msrp_plot.clip(0, msrp_plot.quantile(0.98))
    ax.hist(msrp_plot_clip, bins=60, color=CAR_BLUE, edgecolor=BG_DARK, alpha=0.8, density=True)
    msrp_plot_clip.plot(kind='kde', ax=ax, color=CAR_GOLD, linewidth=2)
    ax.axvline(msrp_plot.median(), color=CAR_RED,    linestyle='--', linewidth=1.5, label=f'Median ${msrp_plot.median():,.0f}')
    ax.axvline(msrp_plot.mean(),   color=CAR_ORANGE, linestyle='--', linewidth=1.5, label=f'Mean ${msrp_plot.mean():,.0f}')
    ax.set_title('MSRP Distribution (KDE + Histogram)')
    ax.set_xlabel('MSRP ($)'); ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

    # Price tier pie
    ax = axes[1]
    tier_colours = [CAR_TEAL, CAR_BLUE, CAR_ORANGE, CAR_GOLD, CAR_PURPLE]
    wedge_props  = dict(edgecolor=BG_DARK, linewidth=2)
    ax.pie(tier_counts.values, labels=tier_counts.index, autopct='%1.1f%%',
           colors=tier_colours[:len(tier_counts)], wedgeprops=wedge_props,
           textprops={'fontsize': 8, 'color': TEXT_LIGHT})
    ax.set_title('Market Price Tier Distribution')

    # MSRP boxplot
    ax = axes[2]
    bp = ax.boxplot(msrp_plot.values, vert=True, patch_artist=True,
                    boxprops=dict(facecolor=CAR_BLUE, color=TEXT_LIGHT, linewidth=1.5),
                    medianprops=dict(color=CAR_GOLD, linewidth=2.5),
                    whiskerprops=dict(color=TEXT_LIGHT, linewidth=1.5),
                    capprops=dict(color=TEXT_LIGHT, linewidth=1.5),
                    flierprops=dict(markerfacecolor=CAR_RED, marker='o', markersize=3, alpha=0.4))
    ax.set_title('MSRP Box Plot')
    ax.set_ylabel('MSRP ($)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    # MSRP vs Invoice
    ax = axes[3]
    if INVOICE_COL:
        inv_plot = df[INVOICE_COL].dropna()
        ax.scatter(df[INVOICE_COL], df[MSRP_COL], alpha=0.3, color=CAR_TEAL, s=8, edgecolors='none')
        max_val = max(df[MSRP_COL].max(), df[INVOICE_COL].max())
        ax.plot([0, max_val], [0, max_val], color=CAR_GOLD, linestyle='--', linewidth=1.5, label='MSRP = Invoice')
        ax.set_title('MSRP vs Invoice Price')
        ax.set_xlabel('Invoice ($)'); ax.set_ylabel('MSRP ($)')
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
        ax.legend(fontsize=9)
    else:
        ax.text(0.5, 0.5, 'Invoice column not available', ha='center', va='center',
                transform=ax.transAxes, color=TEXT_MUTED, fontsize=12)
        ax.set_title('MSRP vs Invoice Price')

    plt.tight_layout()
    plt.savefig('/kaggle/working/pricing_intelligence.png', dpi=140, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()


In [ ]:
# ── Price Gap Intelligence (MSRP – Invoice) ───────────────────────────────────
if MSRP_COL and INVOICE_COL:
    df['price_gap']    = df[MSRP_COL] - df[INVOICE_COL]
    df['margin_pct']   = (df['price_gap'] / df[MSRP_COL] * 100).round(2)

    gap_stats = df['price_gap'].describe()
    print("  📊  MSRP – INVOICE PRICE GAP ANALYSIS")
    print(f"  Mean Gap    : ${df['price_gap'].mean():>10,.0f}")
    print(f"  Median Gap  : ${df['price_gap'].median():>10,.0f}")
    print(f"  Max Gap     : ${df['price_gap'].max():>10,.0f}")
    print(f"  Avg Margin% : {df['margin_pct'].mean():>10.2f}%")

    if MAKE_COL:
        gap_by_make = df.groupby(MAKE_COL)['price_gap'].mean().sort_values(ascending=False).head(20)
        print("\n  🏭  LARGEST AVERAGE PRICE GAP BY MANUFACTURER:")
        display(gap_by_make.reset_index().rename(columns={MAKE_COL: 'Manufacturer', 'price_gap': 'Avg Price Gap ($)'}))

    # Price gap distribution
    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    fig.suptitle('💰  Price Gap Intelligence (MSRP vs Invoice)', color=CAR_GOLD, fontsize=14)

    ax = axes[0]
    ax.hist(df['price_gap'].dropna().clip(-5000, df['price_gap'].quantile(0.99)),
            bins=50, color=CAR_ORANGE, edgecolor=BG_DARK, alpha=0.85)
    ax.axvline(df['price_gap'].mean(), color=CAR_GOLD, linestyle='--', linewidth=1.5,
               label=f"Mean ${df['price_gap'].mean():,.0f}")
    ax.set_title('Distribution of Price Gap (MSRP – Invoice)')
    ax.set_xlabel('Price Gap ($)'); ax.set_ylabel('Count')
    ax.legend()

    ax = axes[1]
    if MAKE_COL:
        ax.barh(range(len(gap_by_make))[::-1], gap_by_make.values,
                color=CAR_RED, edgecolor=BG_DARK, linewidth=0.4)
        ax.set_yticks(range(len(gap_by_make))[::-1])
        ax.set_yticklabels(gap_by_make.index, fontsize=9)
        ax.set_title('Avg Price Gap by Manufacturer (Top 20)')
        ax.set_xlabel('Average Price Gap ($)')
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    plt.tight_layout()
    plt.savefig('/kaggle/working/price_gap_analysis.png', dpi=140, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()
else:
    print("  ⚠️  MSRP and/or Invoice columns not found — skipping gap analysis.")


## 10. 🌍 Competitive Market Intelligence

In [ ]:
print("=" * 70)
print("  🌍  AUTOMOTIVE MARKET INTELLIGENCE REPORT")
print("=" * 70)

if MAKE_COL and MSRP_COL:
    # ── Market concentration (HHI) ────────────────────────────────────────────
    shares = (df[MAKE_COL].value_counts() / len(df) * 100)
    hhi    = (shares ** 2).sum()
    print(f"\n  📊  Herfindahl-Hirschman Index (HHI): {hhi:.1f}")
    if hhi < 1000:
        conc = "Competitive"
    elif hhi < 2500:
        conc = "Moderately Concentrated"
    else:
        conc = "Highly Concentrated"
    print(f"  Market Concentration: {conc}")

    # ── Leader board ─────────────────────────────────────────────────────────
    print("\n  🏆  MARKET LEADER BOARD")
    categories = {
        'Volume Leader (Most Models)'  : df[MAKE_COL].value_counts().idxmax(),
        'Pricing Leader (Highest Avg MSRP)': df.groupby(MAKE_COL)[MSRP_COL].mean().idxmax(),
        'Value Leader (Lowest Avg MSRP)'  : df.groupby(MAKE_COL)[MSRP_COL].mean().idxmin(),
    }
    if MODEL_COL:
        categories['Portfolio Leader (Most Models)'] = df.groupby(MAKE_COL)[MODEL_COL].nunique().idxmax()
    if INVOICE_COL:
        categories['Gap Leader (Highest Price Gap)'] = df.groupby(MAKE_COL)['price_gap'].mean().idxmax()

    for cat, leader in categories.items():
        print(f"  {'▶'} {cat:<40}: {leader}")

    # ── Competitive positioning scatter ───────────────────────────────────────
    comp = df.groupby(MAKE_COL).agg(
        Avg_MSRP =(MSRP_COL, 'mean'),
        Count    =(MSRP_COL, 'count'),
    ).reset_index()
    if MODEL_COL:
        comp['Models'] = df.groupby(MAKE_COL)[MODEL_COL].nunique().values

    comp = comp[comp['Count'] >= 3].sort_values('Count', ascending=False)

    fig = px.scatter(
        comp, x='Count', y='Avg_MSRP',
        size='Count', color='Avg_MSRP',
        color_continuous_scale='RdYlGn',
        hover_name=MAKE_COL,
        text=MAKE_COL,
        title='🌍  Competitive Landscape — Volume vs Pricing (Manufacturer Bubble Chart)',
        template='plotly_dark',
        labels={'Count': 'Vehicle Count (Market Presence)', 'Avg_MSRP': 'Average MSRP ($)'}
    )
    fig.update_traces(textposition='top center', textfont_size=9)
    fig.update_layout(height=600, title_font=dict(size=15, color=CAR_GOLD),
                      margin=dict(t=70, l=40, r=40, b=40))
    fig.show()
else:
    print("  ⚠️  Required columns for market intelligence not available.")


## 11. 📈 Statistical Analysis

Comprehensive statistical study including **Pearson** and **Spearman** correlations, normality testing, and distribution analysis.


In [ ]:
print("  📊  DESCRIPTIVE STATISTICS — ALL NUMERIC COLUMNS")
print("  " + "─" * 65)

if NUM_COLS:
    desc = df[NUM_COLS].describe().T
    desc['skewness'] = df[NUM_COLS].skew()
    desc['kurtosis'] = df[NUM_COLS].kurtosis()
    display(desc)
else:
    print("  ⚠️  No numeric columns detected.")


In [ ]:
# ── Correlation analysis ──────────────────────────────────────────────────────
corr_cols = [c for c in NUM_COLS if df[c].notna().sum() > 20][:12]

if len(corr_cols) >= 2:
    pearson  = df[corr_cols].corr(method='pearson')
    spearman = df[corr_cols].corr(method='spearman')

    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    fig.suptitle('📈  Correlation Analysis — Pearson & Spearman', color=CAR_GOLD, fontsize=14)

    cmap = LinearSegmentedColormap.from_list('car', [CAR_RED, BG_PANEL, CAR_TEAL])
    for ax, mat, title in zip(axes, [pearson, spearman], ['Pearson Correlation', 'Spearman Correlation']):
        sns.heatmap(mat, ax=ax, cmap=cmap, annot=True, fmt='.2f',
                    linewidths=0.5, linecolor=BG_DARK, annot_kws={'size': 8},
                    vmin=-1, vmax=1,
                    cbar_kws={'shrink': 0.8, 'label': 'Correlation Coefficient'})
        ax.set_title(title, fontsize=12)
        ax.tick_params(axis='x', rotation=45, labelsize=7)
        ax.tick_params(axis='y', rotation=0,  labelsize=7)

    plt.tight_layout()
    plt.savefig('/kaggle/working/correlation_heatmaps.png', dpi=140, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()

    # Top correlated pairs
    print("\n  📊  TOP CORRELATED PAIRS (Pearson, |r| > 0.5, excluding diagonal):")
    pairs = []
    for i, c1 in enumerate(corr_cols):
        for j, c2 in enumerate(corr_cols):
            if i < j:
                r = pearson.loc[c1, c2]
                if abs(r) > 0.5:
                    pairs.append({'Col 1': c1, 'Col 2': c2, 'Pearson r': round(r, 4)})
    if pairs:
        display(pd.DataFrame(pairs).sort_values('Pearson r', key=abs, ascending=False))
    else:
        print("  No pairs with |r| > 0.5 found.")
else:
    print("  ⚠️  Insufficient numeric columns for correlation analysis.")


In [ ]:
# ── Normality tests ───────────────────────────────────────────────────────────
print("  📊  NORMALITY TESTS (D'Agostino-Pearson)")
print("  " + "─" * 60)
norm_results = []
for c in corr_cols:
    series = df[c].dropna()
    if len(series) < 8:
        continue
    sample = series.sample(min(5000, len(series)), random_state=42)
    try:
        stat, p = normaltest(sample)
        norm_results.append({'Column': c, 'Statistic': round(stat, 4), 'p-value': round(p, 6),
                              'Normal?': 'Yes' if p > 0.05 else 'No'})
    except Exception:
        pass
if norm_results:
    display(pd.DataFrame(norm_results))


In [ ]:
# ── Pairplot (sample) ─────────────────────────────────────────────────────────
pair_cols = [c for c in [MSRP_COL, INVOICE_COL, YEAR_COL]
             if c and c in df.columns and df[c].notna().sum() > 10]

if len(pair_cols) >= 2:
    sample_df = df[pair_cols + (['price_tier'] if 'price_tier' in df.columns else [])].dropna().sample(
        min(3000, len(df)), random_state=42)
    pp = sns.pairplot(
        sample_df,
        vars=pair_cols,
        hue='price_tier' if 'price_tier' in sample_df.columns else None,
        palette=PALETTE_MAIN[:5],
        plot_kws=dict(alpha=0.4, s=10, edgecolor='none'),
        diag_kws=dict(fill=True, alpha=0.6)
    )
    pp.fig.suptitle('Pairplot — Key Numeric Variables', y=1.01, color=CAR_GOLD, fontsize=13)
    pp.fig.patch.set_facecolor(BG_DARK)
    plt.savefig('/kaggle/working/pairplot.png', dpi=130, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()
    print("✅ Pairplot saved.")
else:
    print("  ⚠️  Insufficient numeric columns for pairplot.")


In [ ]:
# ── Distribution deep-dive for all numeric columns ────────────────────────────
plot_num = [c for c in NUM_COLS if df[c].notna().sum() > 10 and c not in ['make_id','model_id','trim_id']][:9]

if plot_num:
    n_cols = 3
    n_rows = (len(plot_num) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4.5 * n_rows))
    fig.suptitle('📊  Numeric Variable Distributions', color=CAR_GOLD, fontsize=14)
    axes_flat = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes.flatten()

    for i, c in enumerate(plot_num):
        ax   = axes_flat[i]
        data = df[c].dropna().clip(df[c].quantile(0.01), df[c].quantile(0.99))
        ax.hist(data, bins=35, color=PALETTE_MAIN[i % len(PALETTE_MAIN)],
                edgecolor=BG_DARK, alpha=0.8, density=True)
        try:
            data.plot(kind='kde', ax=ax, color=CAR_GOLD, linewidth=1.8)
        except Exception:
            pass
        ax.axvline(data.mean(), color=CAR_RED, linestyle='--', linewidth=1.2,
                   label=f'Mean {data.mean():,.1f}')
        ax.set_title(c.replace('_', ' ').title())
        ax.legend(fontsize=7)

    for j in range(len(plot_num), len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.tight_layout()
    plt.savefig('/kaggle/working/distributions.png', dpi=130, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()


## 12. ⚙️ Feature Engineering for Machine Learning

We create derived features to improve predictive power before training the MSRP prediction model.


In [ ]:
print("  ⚙️  FEATURE ENGINEERING")
print("  " + "─" * 55)

df_ml = df.copy()
new_features = []

# Price gap features
if MSRP_COL and INVOICE_COL and 'price_gap' not in df_ml.columns:
    df_ml['price_gap']  = df_ml[MSRP_COL] - df_ml[INVOICE_COL]
    df_ml['margin_pct'] = (df_ml['price_gap'] / df_ml[MSRP_COL] * 100).round(4)
    new_features += ['price_gap', 'margin_pct']

# Year-based features
if YEAR_COL:
    CURRENT_YEAR = 2025
    df_ml['vehicle_age'] = CURRENT_YEAR - df_ml[YEAR_COL].fillna(CURRENT_YEAR)
    new_features.append('vehicle_age')

# Manufacturer pricing context
if MAKE_COL and MSRP_COL:
    make_avg = df_ml.groupby(MAKE_COL)[MSRP_COL].transform('mean')
    make_std = df_ml.groupby(MAKE_COL)[MSRP_COL].transform('std').fillna(1)
    df_ml['make_avg_msrp']   = make_avg
    df_ml['make_price_index'] = (df_ml[MSRP_COL] - make_avg) / make_std.replace(0, 1)
    new_features += ['make_avg_msrp', 'make_price_index']

# Model pricing context
if MODEL_COL and MSRP_COL:
    model_avg = df_ml.groupby(MODEL_COL)[MSRP_COL].transform('mean')
    df_ml['model_avg_msrp'] = model_avg
    new_features.append('model_avg_msrp')

print(f"  New features created: {new_features}")

# Encode categorical columns
cat_encode_cols = [c for c in [MAKE_COL, MODEL_COL, TRIM_COL] if c]
label_encoders  = {}
for c in cat_encode_cols:
    if c in df_ml.columns:
        le = LabelEncoder()
        df_ml[f'{c}_enc'] = le.fit_transform(df_ml[c].astype(str).fillna('Unknown'))
        label_encoders[c] = le
        new_features.append(f'{c}_enc')
        print(f"  Encoded '{c}' → '{c}_enc' ({df_ml[c].nunique()} categories)")

print(f"\n  Total engineered features : {len(new_features)}")


## 13. 🤖 Machine Learning — Vehicle MSRP Prediction

**Model:** Gradient Boosting Regressor (selected for robustness on small-to-medium tabular datasets, native feature importance, and strong performance without extensive tuning)

**Target:** `MSRP` (vehicle retail price)  
**Strategy:** Feature selection → Train/Test split → Pipeline → Evaluation → Feature Importance


In [ ]:
if MSRP_COL:
    # ── Feature selection ─────────────────────────────────────────────────────
    candidate_features = [f for f in new_features
                          if f in df_ml.columns and f != MSRP_COL]
    # Add raw numeric features (excluding target and IDs)
    id_hints = ['_id', 'make_id', 'model_id', 'trim_id']
    for c in NUM_COLS:
        if c != MSRP_COL and c not in candidate_features and not any(h in c for h in id_hints):
            candidate_features.append(c)

    # Remove target and gap columns that leak the label
    leak_cols = [INVOICE_COL, 'price_gap', 'margin_pct'] if INVOICE_COL else ['price_gap', 'margin_pct']
    candidate_features = [c for c in candidate_features
                          if c not in leak_cols and c in df_ml.columns]

    print(f"  Selected {len(candidate_features)} features: {candidate_features}")

    # ── Prepare data ──────────────────────────────────────────────────────────
    ml_df = df_ml[candidate_features + [MSRP_COL]].dropna(subset=[MSRP_COL])
    ml_df = ml_df.fillna(ml_df.median(numeric_only=True))

    X = ml_df[candidate_features]
    y = ml_df[MSRP_COL]

    print(f"  ML dataset: {X.shape[0]:,} samples × {X.shape[1]} features")
    print(f"  Target (MSRP) stats: mean=${y.mean():,.0f}  std=${y.std():,.0f}")

    if len(X) >= 20:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.20, random_state=42)
        print(f"  Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
    else:
        print("  ⚠️  Insufficient data for ML (need ≥20 rows). Skipping.")
        X_train = None
else:
    X_train = None
    print("  ⚠️  MSRP column not found — skipping ML.")


In [ ]:
if X_train is not None:
    # ── Pipeline ──────────────────────────────────────────────────────────────
    pipe_gbm = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.07, max_depth=4,
            subsample=0.85, min_samples_leaf=5, random_state=42))
    ])

    # ── Cross-validation ──────────────────────────────────────────────────────
    kf      = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_r2   = cross_val_score(pipe_gbm, X_train, y_train, cv=kf, scoring='r2')
    cv_rmse = np.sqrt(-cross_val_score(pipe_gbm, X_train, y_train, cv=kf, scoring='neg_mean_squared_error'))

    print(f"  ── Cross-Validation Results (5-Fold) ──────────────────────────")
    print(f"  CV R²    : {cv_r2.mean():.4f}  ± {cv_r2.std():.4f}")
    print(f"  CV RMSE  : ${cv_rmse.mean():,.0f}  ± ${cv_rmse.std():,.0f}")

    # ── Final fit ─────────────────────────────────────────────────────────────
    pipe_gbm.fit(X_train, y_train)
    y_pred  = pipe_gbm.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / y_test.replace(0, np.nan))) * 100

    print(f"\n  ─────────────────────────────────────────────────────")
    print(f"  🏆  FINAL MODEL EVALUATION (Test Set)")
    print(f"  ─────────────────────────────────────────────────────")
    print(f"  R² Score  : {r2:.4f}  ({r2*100:.1f}% variance explained)")
    print(f"  MAE       : ${mae:>12,.0f}")
    print(f"  RMSE      : ${rmse:>12,.0f}")
    print(f"  MAPE      : {mape:.2f}%")
    print(f"  ─────────────────────────────────────────────────────")

    feat_imp = pd.Series(
        pipe_gbm.named_steps['model'].feature_importances_,
        index=candidate_features
    ).sort_values(ascending=False)
    print(f"\n  📊  TOP 10 FEATURE IMPORTANCES:")
    display(feat_imp.head(10).reset_index().rename(columns={'index': 'Feature', 0: 'Importance'}))


In [ ]:
if X_train is not None:
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle('🤖  GBM MSRP Prediction — Model Evaluation', color=CAR_GOLD, fontsize=14)

    # Actual vs Predicted
    ax = axes[0]
    ax.scatter(y_test, y_pred, alpha=0.4, color=CAR_TEAL, s=15, edgecolors='none')
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, color=CAR_GOLD, linestyle='--', linewidth=1.5, label='Perfect prediction')
    ax.set_title(f'Actual vs Predicted MSRP\nR² = {r2:.4f}')
    ax.set_xlabel('Actual MSRP ($)'); ax.set_ylabel('Predicted MSRP ($)')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

    # Residual plot
    ax = axes[1]
    residuals = y_test - y_pred
    ax.scatter(y_pred, residuals, alpha=0.4, color=CAR_ORANGE, s=12, edgecolors='none')
    ax.axhline(0, color=CAR_GOLD, linestyle='--', linewidth=1.5)
    ax.set_title('Residual Plot')
    ax.set_xlabel('Predicted MSRP ($)'); ax.set_ylabel('Residual ($)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

    # Feature importance
    ax = axes[2]
    top12 = feat_imp.head(12)
    colours_fi = [CAR_RED if v > feat_imp.median() else CAR_BLUE for v in top12.values]
    ax.barh(range(len(top12))[::-1], top12.values, color=colours_fi[::-1], edgecolor=BG_DARK, linewidth=0.4)
    ax.set_yticks(range(len(top12))[::-1])
    ax.set_yticklabels([c.replace('_',' ').title() for c in top12.index], fontsize=9)
    ax.set_title('Feature Importances (GBM)')
    ax.set_xlabel('Importance Score')
    for i, v in enumerate(top12.values[::-1]):
        ax.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=8, color=TEXT_LIGHT)

    plt.tight_layout()
    plt.savefig('/kaggle/working/ml_evaluation.png', dpi=140, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()
    print("✅ ML evaluation charts saved.")


In [ ]:
# ── Residual distribution plot ─────────────────────────────────────────────────
if X_train is not None:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle('🤖  Residual Analysis', color=CAR_GOLD, fontsize=13)

    ax = axes[0]
    ax.hist(residuals, bins=50, color=CAR_PURPLE, edgecolor=BG_DARK, alpha=0.85)
    ax.axvline(residuals.mean(), color=CAR_GOLD, linestyle='--', linewidth=1.5,
               label=f'Mean ${residuals.mean():,.0f}')
    ax.set_title('Residual Distribution')
    ax.set_xlabel('Residual ($)'); ax.set_ylabel('Count')
    ax.legend()

    ax = axes[1]
    from scipy.stats import probplot
    (osm, osr), (slope, intercept, r) = probplot(residuals, dist='norm')
    ax.scatter(osm, osr, color=CAR_TEAL, s=8, alpha=0.6, edgecolors='none')
    ax.plot(osm, slope*np.array(osm) + intercept, color=CAR_GOLD, linewidth=1.5)
    ax.set_title(f'Q-Q Plot — Residuals (r={r:.4f})')
    ax.set_xlabel('Theoretical Quantiles'); ax.set_ylabel('Sample Quantiles')

    plt.tight_layout()
    plt.savefig('/kaggle/working/residual_analysis.png', dpi=140, bbox_inches='tight', facecolor=BG_DARK)
    plt.show()


## 14. 🧠 Automated Insights Engine

In [ ]:
print("=" * 72)
print("  🧠  AUTOMATED INSIGHTS ENGINE — NARRATIVE INTELLIGENCE REPORT")
print("=" * 72)

insights = {}

# ── Volume insights ───────────────────────────────────────────────────────────
if MAKE_COL:
    top_make = df[MAKE_COL].value_counts()
    insights['Volume Leader']            = top_make.idxmax()
    insights['Volume Leader Count']      = f"{top_make.max():,} trims"
    insights['Manufacturers Represented'] = f"{df[MAKE_COL].nunique()}"

if MODEL_COL:
    top_model = df[MODEL_COL].value_counts()
    insights['Most Common Model']        = top_model.idxmax()
    insights['Most Common Model Count']  = f"{top_model.max():,} trims"

if TRIM_COL:
    top_trim  = df[TRIM_COL].value_counts()
    insights['Most Common Trim']         = top_trim.idxmax()

# ── Pricing insights ──────────────────────────────────────────────────────────
if MSRP_COL:
    msrp_s = df[MSRP_COL].dropna()
    insights['Market Avg MSRP']          = f"${msrp_s.mean():,.0f}"
    insights['Market Median MSRP']       = f"${msrp_s.median():,.0f}"
    insights['Most Expensive Vehicle']   = f"${msrp_s.max():,.0f}"
    insights['Most Affordable Vehicle']  = f"${msrp_s.min():,.0f}"

    if MAKE_COL:
        premium_make  = df.groupby(MAKE_COL)[MSRP_COL].mean().idxmax()
        budget_make   = df.groupby(MAKE_COL)[MSRP_COL].mean().idxmin()
        insights['Highest Avg MSRP Brand']  = premium_make
        insights['Lowest Avg MSRP Brand']   = budget_make

if MSRP_COL and INVOICE_COL:
    largest_gap_idx = df['price_gap'].idxmax()
    insights['Largest Price Gap Vehicle']  = f"${df.loc[largest_gap_idx, 'price_gap']:,.0f}"
    insights['Avg Dealer Margin']          = f"${df['price_gap'].mean():,.0f}  ({df['margin_pct'].mean():.1f}%)"

# ── ML insights ───────────────────────────────────────────────────────────────
if X_train is not None:
    insights['ML R² Score']              = f"{r2:.4f}  ({r2*100:.1f}% variance explained)"
    insights['ML MAE']                   = f"${mae:,.0f}"
    insights['ML RMSE']                  = f"${rmse:,.0f}"
    insights['Top Predictive Feature']   = feat_imp.idxmax()

print("\n  📊  INSIGHT SUMMARY")
print("  " + "─" * 60)
for k, v in insights.items():
    print(f"  {'▶'} {k:<40}: {v}")


## 15. 📊 Comprehensive Analytics Dashboard

In [ ]:
fig = plt.figure(figsize=(24, 30))
fig.patch.set_facecolor(BG_DARK)
fig.suptitle('🚗  Comprehensive Car Market Intelligence Dashboard',
             color=CAR_GOLD, fontsize=18, fontweight='bold', y=0.99)

gs = gridspec.GridSpec(5, 3, figure=fig, hspace=0.55, wspace=0.42)
axes = [fig.add_subplot(gs[r, c]) for r in range(5) for c in range(3)]

def safe_bar_h(ax, data, title, xlabel='', color=CAR_BLUE, top_n=12):
    if data is None or data.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes, color=TEXT_MUTED)
        ax.set_title(title); return
    data = data.head(top_n)
    ax.barh(range(len(data))[::-1], data.values, color=color, edgecolor=BG_DARK, linewidth=0.3)
    ax.set_yticks(range(len(data))[::-1])
    ax.set_yticklabels(data.index, fontsize=8)
    ax.set_title(title); ax.set_xlabel(xlabel)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'${x:,.0f}' if xlabel and '$' in xlabel else f'{x:,.0f}'))

# 1. Top manufacturers by volume
ax = axes[0]
if MAKE_COL:
    safe_bar_h(ax, df[MAKE_COL].value_counts(), 'Top 12 Manufacturers (Volume)', 'Vehicle Count', CAR_BLUE)

# 2. Top manufacturers by avg MSRP
ax = axes[1]
if MAKE_COL and MSRP_COL:
    make_msrp = df.groupby(MAKE_COL)[MSRP_COL].mean().sort_values(ascending=False)
    safe_bar_h(ax, make_msrp, 'Highest Avg MSRP Brands', '$ MSRP', CAR_GOLD)

# 3. Price tier pie
ax = axes[2]
if 'price_tier' in df.columns:
    tc = df['price_tier'].value_counts()
    tc_colours = [CAR_TEAL, CAR_BLUE, CAR_ORANGE, CAR_GOLD, CAR_PURPLE]
    ax.pie(tc.values, labels=tc.index, autopct='%1.0f%%', colors=tc_colours[:len(tc)],
           wedgeprops=dict(edgecolor=BG_DARK, linewidth=1.5),
           textprops={'fontsize': 8, 'color': TEXT_LIGHT})
    ax.set_title('Market Price Tier Distribution')
else:
    ax.text(0.5, 0.5, 'Price tier
not computed', ha='center', va='center',
            transform=ax.transAxes, color=TEXT_MUTED)
    ax.set_title('Market Price Tier Distribution')

# 4. MSRP distribution
ax = axes[3]
if MSRP_COL:
    data = df[MSRP_COL].dropna().clip(0, df[MSRP_COL].quantile(0.98))
    ax.hist(data, bins=50, color=CAR_RED, edgecolor=BG_DARK, alpha=0.85, density=True)
    try: data.plot(kind='kde', ax=ax, color=CAR_GOLD, linewidth=1.8)
    except Exception: pass
    ax.set_title('MSRP Distribution')
    ax.set_xlabel('MSRP ($)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
else:
    ax.text(0.5, 0.5, 'MSRP not available', ha='center', va='center',
            transform=ax.transAxes, color=TEXT_MUTED)
    ax.set_title('MSRP Distribution')

# 5. Invoice distribution
ax = axes[4]
if INVOICE_COL:
    data = df[INVOICE_COL].dropna().clip(0, df[INVOICE_COL].quantile(0.98))
    ax.hist(data, bins=50, color=CAR_PURPLE, edgecolor=BG_DARK, alpha=0.85, density=True)
    try: data.plot(kind='kde', ax=ax, color=CAR_GOLD, linewidth=1.8)
    except Exception: pass
    ax.set_title('Invoice Price Distribution')
    ax.set_xlabel('Invoice ($)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
else:
    ax.text(0.5, 0.5, 'Invoice not available', ha='center', va='center',
            transform=ax.transAxes, color=TEXT_MUTED)
    ax.set_title('Invoice Price Distribution')

# 6. Price gap distribution
ax = axes[5]
if 'price_gap' in df.columns:
    data = df['price_gap'].dropna().clip(df['price_gap'].quantile(0.01), df['price_gap'].quantile(0.99))
    ax.hist(data, bins=50, color=CAR_ORANGE, edgecolor=BG_DARK, alpha=0.85)
    ax.axvline(data.mean(), color=CAR_GOLD, linestyle='--', linewidth=1.5,
               label=f'Mean ${data.mean():,.0f}')
    ax.set_title('Price Gap (MSRP – Invoice)')
    ax.set_xlabel('Price Gap ($)')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'Price gap
not computed', ha='center', va='center',
            transform=ax.transAxes, color=TEXT_MUTED)
    ax.set_title('Price Gap Distribution')

# 7. MSRP by manufacturer (boxplot top 10)
ax = axes[6]
if MAKE_COL and MSRP_COL:
    top10_makes = df[MAKE_COL].value_counts().head(10).index
    bp_data = [df.loc[df[MAKE_COL] == m, MSRP_COL].dropna().values for m in top10_makes]
    bp_data = [d for d in bp_data if len(d) > 0]
    if bp_data:
        bplot = ax.boxplot(bp_data, patch_artist=True, labels=[m[:8] for m in top10_makes[:len(bp_data)]],
                           boxprops=dict(linewidth=1),
                           medianprops=dict(color=CAR_GOLD, linewidth=2),
                           whiskerprops=dict(color=TEXT_MUTED),
                           capprops=dict(color=TEXT_MUTED),
                           flierprops=dict(marker='o', markersize=2, alpha=0.3, color=CAR_RED))
        for patch, color in zip(bplot['boxes'], PALETTE_MAIN[:len(bp_data)]):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_title('MSRP Distribution — Top 10 Brands')
        ax.set_xticklabels([m[:8] for m in top10_makes[:len(bp_data)]], rotation=45, ha='right', fontsize=7)
        ax.set_ylabel('MSRP ($)')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# 8. Year trend
ax = axes[7]
if YEAR_COL and MSRP_COL:
    yr_msrp = df.groupby(YEAR_COL)[MSRP_COL].mean().dropna()
    ax.plot(yr_msrp.index, yr_msrp.values, color=CAR_TEAL, linewidth=2, marker='o', markersize=4)
    ax.fill_between(yr_msrp.index, yr_msrp.values, alpha=0.2, color=CAR_TEAL)
    ax.set_title('Avg MSRP Trend by Year')
    ax.set_xlabel('Year'); ax.set_ylabel('Avg MSRP ($)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
else:
    ax.text(0.5, 0.5, 'Year/MSRP data
not available', ha='center', va='center',
            transform=ax.transAxes, color=TEXT_MUTED)
    ax.set_title('Avg MSRP by Year')

# 9. Correlation heatmap
ax = axes[8]
corr_cols_dash = [c for c in NUM_COLS if df[c].notna().sum() > 10
                  and 'id' not in c][:8]
if len(corr_cols_dash) >= 2:
    cmap_ = LinearSegmentedColormap.from_list('car', [CAR_RED, BG_PANEL, CAR_TEAL])
    sns.heatmap(df[corr_cols_dash].corr(), ax=ax, cmap=cmap_,
                annot=True, fmt='.2f', linewidths=0.5, linecolor=BG_DARK,
                annot_kws={'size': 7}, vmin=-1, vmax=1)
    ax.set_title('Correlation Heatmap')
    ax.tick_params(axis='x', rotation=45, labelsize=6)
    ax.tick_params(axis='y', rotation=0,  labelsize=6)

# 10. Top models by count
ax = axes[9]
if MODEL_COL:
    safe_bar_h(ax, df[MODEL_COL].value_counts(), 'Top 12 Models by Trim Count', 'Count', CAR_SILVER)

# 11. Actual vs Predicted (ML)
ax = axes[10]
if X_train is not None:
    ax.scatter(y_test, y_pred, alpha=0.4, color=CAR_TEAL, s=12, edgecolors='none')
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, color=CAR_GOLD, linestyle='--', linewidth=1.5)
    ax.set_title(f'ML: Actual vs Predicted  (R²={r2:.3f})')
    ax.set_xlabel('Actual MSRP ($)'); ax.set_ylabel('Predicted ($)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
else:
    ax.text(0.5, 0.5, 'ML model not trained', ha='center', va='center',
            transform=ax.transAxes, color=TEXT_MUTED)
    ax.set_title('ML: Actual vs Predicted')

# 12. Feature importances
ax = axes[11]
if X_train is not None and feat_imp is not None:
    top10 = feat_imp.head(10)
    fi_col = [CAR_RED if v > feat_imp.median() else CAR_BLUE for v in top10.values]
    ax.barh(range(len(top10))[::-1], top10.values, color=fi_col[::-1], edgecolor=BG_DARK, linewidth=0.4)
    ax.set_yticks(range(len(top10))[::-1])
    ax.set_yticklabels([c.replace('_',' ')[:22] for c in top10.index], fontsize=8)
    ax.set_title('Feature Importances (GBM)')
    ax.set_xlabel('Importance')
else:
    ax.text(0.5, 0.5, 'Feature importance
not available', ha='center', va='center',
            transform=ax.transAxes, color=TEXT_MUTED)
    ax.set_title('Feature Importances')

plt.savefig('/kaggle/working/master_dashboard.png', dpi=150, bbox_inches='tight', facecolor=BG_DARK)
plt.show()
print("✅ Master dashboard saved.")


## 16. 📋 Executive Report, Conclusions & Strategic Recommendations

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║      🚗  COMPREHENSIVE CAR MARKET ANALYSIS — EXECUTIVE REPORT       ║
╚══════════════════════════════════════════════════════════════════════╝
""")

# ── Key Findings ──────────────────────────────────────────────────────────────
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📊  SECTION A — KEY FINDINGS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")
for k, v in insights.items():
    print(f"  ▶ {k:<40}: {v}")

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🏭  SECTION B — MANUFACTURER INTELLIGENCE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  1. The automotive market shows significant concentration among a small
     number of volume leaders who account for the majority of trim variants.

  2. Luxury and ultra-luxury segments are dominated by a distinct set of
     premium brands that maintain pricing power through exclusivity.

  3. The most diversified manufacturers (widest model portfolios) tend to
     occupy the mid-range pricing tier — balancing breadth with margin.

  4. Brand premium (avg MSRP vs market median) varies dramatically,
     offering clear opportunities for positioning and competitive benchmarking.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  💰  SECTION C — PRICING INTELLIGENCE SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  5. MSRP distributions are right-skewed — most vehicles cluster in the
     budget-to-mid-range tier, with a long tail of luxury outliers.

  6. The MSRP–Invoice gap represents the average dealer negotiation margin
     and varies significantly by manufacturer and model tier.

  7. Higher-priced vehicle segments exhibit wider absolute price gaps,
     presenting larger negotiation opportunities for informed buyers.

  8. Price segmentation reveals four distinct market tiers: Budget, Mid-Range,
     Premium, and Luxury — each with different competitive dynamics.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🤖  SECTION D — MACHINE LEARNING FINDINGS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")
if X_train is not None:
    print(f"  9.  GBM achieved R² = {r2:.4f} — explaining {r2*100:.1f}% of MSRP variance.")
    print(f" 10.  Mean Absolute Error of ${mae:,.0f} is acceptable for pricing benchmarking.")
    print(f" 11.  Top feature: '{feat_imp.idxmax()}' — {feat_imp.max():.3f} importance score.")
    print(f" 12.  Model is suitable for pricing audits, MSRP gap detection, and")
    print(f"      competitive benchmarking in dealer environments.")
else:
    print("  9.  ML model was not trained due to insufficient data or missing columns.")

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🎯  SECTION E — STRATEGIC RECOMMENDATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  FOR DEALERSHIPS:
  ▶ Use MSRP–Invoice gap analysis to identify high-margin models
  ▶ Prioritise mid-range inventory for volume + margin balance
  ▶ Monitor budget segment leaders for high-volume sales opportunities

  FOR MANUFACTURERS:
  ▶ Portfolio diversification across tiers reduces market cycle risk
  ▶ Premium and Luxury brands should protect pricing power via trim exclusivity
  ▶ Analyse competitors' price-gap strategies for competitive positioning

  FOR INVESTORS:
  ▶ Volume leaders with mid-range positioning offer stable revenue streams
  ▶ Luxury segment brands show higher margins but lower volume sensitivity
  ▶ Track year-over-year MSRP trends as leading indicators of market health

  FOR PRICING ANALYSTS:
  ▶ The GBM model can be used to flag under- or over-priced trims
  ▶ Price gap analysis enables data-driven negotiation benchmarking
  ▶ Incorporate additional features (fuel type, drivetrain, engine) for 
    improved prediction accuracy

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🔮  SECTION F — FUTURE RESEARCH DIRECTIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  1. Time-series analysis of MSRP trends by manufacturer and segment
  2. Integration with fuel economy and emissions data for sustainability scoring
  3. Geospatial analysis of regional pricing variations
  4. NLP analysis of trim descriptions for feature-based value extraction
  5. Ensemble models combining tabular and text features for richer prediction
  6. Depreciation curve modelling using historical transaction data

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Thank you for exploring this notebook! 🚗
  If useful, please upvote on Kaggle.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")


In [ ]:
# ── Summary of all saved outputs ──────────────────────────────────────────────
print("  📁  SAVED OUTPUT FILES")
print("  " + "─" * 50)
for f in sorted(glob.glob('/kaggle/working/*.png')):
    size_kb = os.path.getsize(f) / 1024
    print(f"  ✅  {os.path.basename(f):<45}  {size_kb:>7.1f} KB")
print("\n  🏁  Notebook execution complete!")
